# VC · Práctica 5 — Entrenamiento de Modelos Clásicos y Evaluación de DeepFace (FER-2013)

Este notebook **entrena y guarda** los modelos clásicos (PCA / HOG / LBP + KNN / SVM / Stacked) y ejecuta **DeepFace** sobre la muestra estratificada de `test`, guardando sus predicciones y tiempos en `outputs_p5/models/v1_otros/`.

De este modo, el notebook de comparativa (`VC_P5_emociones_FER.ipynb`) solo carga los resultados ya calculados junto con la CNN entrenada (`v1_CNN`).

## 1. Configuración y librerías

In [1]:
import os, time, json, random, warnings
from pathlib import Path
import ctypes, glob, site

_nv = os.path.join(site.getsitepackages()[0], "nvidia")
for _name in ["cusolver/lib/libcusolver.so.11", "curand/lib/libcurand.so.10"]:
    for _p in glob.glob(os.path.join(_nv, _name)):
        ctypes.CDLL(_p, mode=ctypes.RTLD_GLOBAL)

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")
os.environ.setdefault("DEEPFACE_HOME", os.path.abspath("./.deepface_cache"))
os.environ.setdefault("MPLCONFIGDIR", "/tmp/mpl_cache")

import cv2
import numpy as np
import pandas as pd
import joblib
from joblib import Parallel, delayed
from tqdm.auto import tqdm

from skimage.feature import hog, local_binary_pattern
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split, StratifiedShuffleSplit
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC, LinearSVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

warnings.filterwarnings("ignore", category=UserWarning)

# =====================  PARÁMETROS  =====================
DATA_DIR = Path(
    r"/home/cris/Proyectos/computer-vision/image-dataset/prac5_dataset"
)
TRAIN_DIR, TEST_DIR = DATA_DIR / "train", DATA_DIR / "test"

OUT_DIR    = Path("outputs_p5")
OTHERS_DIR = OUT_DIR / "models/v1_otros"
OTHERS_DIR.mkdir(parents=True, exist_ok=True)

SEED     = 42
IMG_SIZE = 48
VAL_SIZE = 0.20
N_JOBS   = -1

# Modelos clásicos
KNN_K_GRID      = [5, 11, 21]
SVM_KIND        = "rbf"      # "rbf" | "linear"
SVM_MAX_TRAIN   = 10_000
SVM_C_GRID      = [1, 10]
RUN_STACKED     = True
STACK_MAX_TRAIN = 5_000

# DeepFace
DEEPFACE_N        = 1000
DEEPFACE_DETECTOR = "skip"

random.seed(SEED); np.random.seed(SEED)
assert TRAIN_DIR.is_dir() and TEST_DIR.is_dir(), f"No encuentro train/ y test/ en {DATA_DIR}"
print("Guardando modelos clásicos y predicciones DeepFace en:", OTHERS_DIR.resolve())

Guardando modelos clásicos y predicciones DeepFace en: /home/cris/Proyectos/computer-vision/P5/outputs_p5/models/v1_otros


## 2. Carga de datos y particiones estratificadas

In [2]:
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp"}

def read_gray(path: Path) -> np.ndarray:
    data = np.fromfile(str(path), dtype=np.uint8)
    img = cv2.imdecode(data, cv2.IMREAD_GRAYSCALE)
    if img is not None and img.shape != (IMG_SIZE, IMG_SIZE):
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
    return img

def load_split(split_dir: Path, class_names=None):
    if class_names is None:
        class_names = sorted(p.name for p in split_dir.iterdir() if p.is_dir())
    X, y = [], []
    for idx, cname in enumerate(class_names):
        files = sorted(f for f in (split_dir / cname).iterdir() if f.suffix.lower() in IMG_EXT)
        for f in tqdm(files, desc=f"{split_dir.name}/{cname}", leave=False):
            img = read_gray(f)
            if img is not None:
                X.append(img); y.append(idx)
    return np.stack(X).astype(np.uint8), np.array(y, dtype=np.int64), class_names

X_full, y_full, CLASS_NAMES = load_split(TRAIN_DIR)
X_test, y_test, _ = load_split(TEST_DIR, CLASS_NAMES)
N_CLASSES = len(CLASS_NAMES)

X_tr, X_val, y_tr, y_val = train_test_split(
    X_full, y_full, test_size=VAL_SIZE, stratify=y_full, random_state=SEED
)

# Muestra estratificada común de test para DeepFace y comparativa
if DEEPFACE_N is not None and DEEPFACE_N < len(y_test):
    sss = StratifiedShuffleSplit(n_splits=1, test_size=DEEPFACE_N, random_state=SEED)
    _, df_idx = next(sss.split(X_test, y_test))
    df_idx = np.sort(df_idx)
else:
    df_idx = np.arange(len(y_test))

np.save(OTHERS_DIR / "deepface_test_idx.npy", df_idx)
json.dump(CLASS_NAMES, open(OTHERS_DIR / "class_names.json", "w"))
print(f"Train: {len(y_tr)} | Val: {len(y_val)} | Test: {len(y_test)} | Muestra DeepFace: {len(df_idx)}")

train/angry:   0%|          | 0/3995 [00:00<?, ?it/s]

train/disgusted:   0%|          | 0/436 [00:00<?, ?it/s]

train/fearful:   0%|          | 0/4097 [00:00<?, ?it/s]

train/happy:   0%|          | 0/7215 [00:00<?, ?it/s]

train/neutral:   0%|          | 0/4965 [00:00<?, ?it/s]

train/sad:   0%|          | 0/4830 [00:00<?, ?it/s]

... (5 outputs left out)

test/neutral:   0%|          | 0/1233 [00:00<?, ?it/s]

test/sad:   0%|          | 0/1247 [00:00<?, ?it/s]

test/surprised:   0%|          | 0/831 [00:00<?, ?it/s]

Train: 22967 | Val: 5742 | Test: 7178 | Muestra DeepFace: 1000


## 3. Extracción de características y entrenamiento de modelos clásicos (PCA, HOG, LBP + KNN / SVM / Stacked)

In [3]:
def extract_hog_single(img):
    return hog(
        img, orientations=9, pixels_per_cell=(8, 8),
        cells_per_block=(2, 2), block_norm="L2-Hys", feature_vector=True
    ).astype(np.float32)

def extract_lbp_single(img, radius=2, n_points=16):
    lbp = local_binary_pattern(img, n_points, radius, method="uniform")
    n_bins = n_points + 2
    hist, _ = np.histogram(lbp.ravel(), bins=n_bins, range=(0, n_bins), density=True)
    return hist.astype(np.float32)

def extract_hog_batch(X, n_jobs=N_JOBS):
    return np.vstack(Parallel(n_jobs=n_jobs)(delayed(extract_hog_single)(im) for im in X))

def extract_lbp_batch(X, n_jobs=N_JOBS):
    return np.vstack(Parallel(n_jobs=n_jobs)(delayed(extract_lbp_single)(im) for im in X))

def stratified_cap(X, y, max_n, seed=SEED):
    if max_n is None or len(y) <= max_n:
        return X, y
    sss = StratifiedShuffleSplit(n_splits=1, train_size=max_n, random_state=seed)
    idx, _ = next(sss.split(X, y))
    return X[idx], y[idx]

print("Extrayendo características PCA / HOG / LBP...")
X_tr_flat  = (X_tr.reshape(len(X_tr), -1) / 255.0).astype(np.float32)
X_val_flat = (X_val.reshape(len(X_val), -1) / 255.0).astype(np.float32)

pca = PCA(n_components=0.95, random_state=SEED).fit(X_tr_flat)
X_tr_pca  = pca.transform(X_tr_flat)
X_val_pca = pca.transform(X_val_flat)
joblib.dump(pca, OTHERS_DIR / "pca_transformer.joblib")

X_tr_hog  = extract_hog_batch(X_tr)
X_val_hog = extract_hog_batch(X_val)

X_tr_lbp  = extract_lbp_batch(X_tr)
X_val_lbp = extract_lbp_batch(X_val)

print(f"Dims -> PCA: {X_tr_pca.shape[1]} | HOG: {X_tr_hog.shape[1]} | LBP: {X_tr_lbp.shape[1]}")

Extrayendo características PCA / HOG / LBP...
Dims -> PCA: 252 | HOG: 900 | LBP: 18


In [4]:
def fit_best_knn(X_train_f, y_train_f, X_val_f, y_val_f):
    best_pipe, best_f1, best_k = None, -1.0, None
    for k in KNN_K_GRID:
        pipe = Pipeline([
            ("scaler", StandardScaler()),
            ("clf", KNeighborsClassifier(n_neighbors=k, weights="distance", n_jobs=N_JOBS))
        ])
        pipe.fit(X_train_f, y_train_f)
        f1 = f1_score(y_val_f, pipe.predict(X_val_f), average="macro")
        if f1 > best_f1:
            best_f1, best_pipe, best_k = f1, pipe, k
    # Dejar n_jobs=1 para inferencia single-thread
    best_pipe.named_steps["clf"].set_params(n_jobs=1)
    return best_pipe, best_k, best_f1

def fit_best_svm(X_train_f, y_train_f, X_val_f, y_val_f):
    X_sub, y_sub = stratified_cap(X_train_f, y_train_f, SVM_MAX_TRAIN if SVM_KIND == "rbf" else None)
    best_pipe, best_f1, best_c = None, -1.0, None
    for c in SVM_C_GRID:
        clf = SVC(C=c, kernel="rbf", class_weight="balanced", probability=True, random_state=SEED) if SVM_KIND == "rbf" \
              else LinearSVC(C=c, class_weight="balanced", random_state=SEED)
        pipe = Pipeline([("scaler", StandardScaler()), ("clf", clf)])
        pipe.fit(X_sub, y_sub)
        f1 = f1_score(y_val_f, pipe.predict(X_val_f), average="macro")
        if f1 > best_f1:
            best_f1, best_pipe, best_c = f1, pipe, c
    return best_pipe, best_c, best_f1

classic_models = {}

for feat_name, (X_t, X_v) in [("PCA", (X_tr_pca, X_val_pca)),
                              ("HOG", (X_tr_hog, X_val_hog)),
                              ("LBP", (X_tr_lbp, X_val_lbp))]:
    knn_m, k_opt, f1_k = fit_best_knn(X_t, y_tr, X_v, y_val)
    classic_models[f"{feat_name}+KNN"] = (knn_m, feat_name)
    print(f"{feat_name}+KNN (k={k_opt}) -> val F1-macro={f1_k:.4f}")

    svm_m, c_opt, f1_s = fit_best_svm(X_t, y_tr, X_v, y_val)
    classic_models[f"{feat_name}+SVM"] = (svm_m, feat_name)
    print(f"{feat_name}+SVM (C={c_opt}) -> val F1-macro={f1_s:.4f}")

if RUN_STACKED:
    # Concatenamos [PCA | HOG | LBP] y entrenamos Stacking sobre subconjunto estratificado
    X_tr_all  = np.hstack([X_tr_pca, X_tr_hog, X_tr_lbp])
    X_val_all = np.hstack([X_val_pca, X_val_hog, X_val_lbp])
    X_st, y_st = stratified_cap(X_tr_all, y_tr, STACK_MAX_TRAIN)

    d_pca, d_hog = X_tr_pca.shape[1], X_tr_hog.shape[1]
    from sklearn.compose import ColumnTransformer
    ct_pca = ColumnTransformer([("sel", "passthrough", slice(0, d_pca))])
    ct_hog = ColumnTransformer([("sel", "passthrough", slice(d_pca, d_pca + d_hog))])
    ct_lbp = ColumnTransformer([("sel", "passthrough", slice(d_pca + d_hog, X_tr_all.shape[1]))])

    estimators = [
        ("pca_svm", Pipeline([("sel", ct_pca), ("sc", StandardScaler()), ("svm", SVC(C=10, kernel="rbf", class_weight="balanced", probability=True, random_state=SEED))])),
        ("hog_svm", Pipeline([("sel", ct_hog), ("sc", StandardScaler()), ("svm", SVC(C=10, kernel="rbf", class_weight="balanced", probability=True, random_state=SEED))])),
        ("lbp_svm", Pipeline([("sel", ct_lbp), ("sc", StandardScaler()), ("svm", SVC(C=10, kernel="rbf", class_weight="balanced", probability=True, random_state=SEED))])),
    ]
    stacked = StackingClassifier(
        estimators=estimators,
        final_estimator=LogisticRegression(max_iter=1000, class_weight="balanced", random_state=SEED),
        cv=3, n_jobs=N_JOBS
    )
    stacked.fit(X_st, y_st)
    stacked.set_params(n_jobs=1)
    f1_st = f1_score(y_val, stacked.predict(X_val_all), average="macro")
    classic_models["Stacked"] = (stacked, "ALL")
    print(f"Stacked -> val F1-macro={f1_st:.4f}")

for mname, (model_obj, feat_type) in classic_models.items():
    fname = mname.replace("+", "_").lower() + ".joblib"
    joblib.dump({"model": model_obj, "feat_type": feat_type, "name": mname}, OTHERS_DIR / fname)
print("Modelos clásicos guardados en:", OTHERS_DIR)

PCA+KNN (k=5) -> val F1-macro=0.3184


/home/cris/miniconda3/envs/VC_P5/lib/python3.11/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/cris/miniconda3/envs/VC_P5/lib/python3.11/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


PCA+SVM (C=10) -> val F1-macro=0.3684
HOG+KNN (k=5) -> val F1-macro=0.4541


/home/cris/miniconda3/envs/VC_P5/lib/python3.11/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/cris/miniconda3/envs/VC_P5/lib/python3.11/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


HOG+SVM (C=10) -> val F1-macro=0.4651
LBP+KNN (k=21) -> val F1-macro=0.2935


/home/cris/miniconda3/envs/VC_P5/lib/python3.11/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/cris/miniconda3/envs/VC_P5/lib/python3.11/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


LBP+SVM (C=10) -> val F1-macro=0.2087


/home/cris/miniconda3/envs/VC_P5/lib/python3.11/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/cris/miniconda3/envs/VC_P5/lib/python3.11/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/cris/miniconda3/envs/VC_P5/lib/python3.11/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/home/cris/miniconda3/envs/VC_P5/lib/python3.11/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probabilit

Stacked -> val F1-macro=0.4040
Modelos clásicos guardados en: outputs_p5/models/v1_otros


## 4. Inferencia con DeepFace sobre la muestra estratificada de test y guardado

In [5]:
from deepface import DeepFace

DEEPFACE_TO_CLASS = {
    "angry": "angry",
    "disgust": "disgusted",
    "fear": "fearful",
    "happy": "happy",
    "neutral": "neutral",
    "sad": "sad",
    "surprise": "surprised",
}
NAME_TO_IDX = {name: i for i, name in enumerate(CLASS_NAMES)}

X_df = X_test[df_idx]
y_df = y_test[df_idx]

# Warm-up para no incluir descarga/carga inicial en el tiempo de inferencia
_ = DeepFace.analyze(
    img_path=cv2.cvtColor(X_df[0], cv2.COLOR_GRAY2RGB),
    actions=["emotion"], detector_backend=DEEPFACE_DETECTOR,
    enforce_detection=False, silent=True
)

df_preds = []
t0 = time.perf_counter()
for img in tqdm(X_df, desc="DeepFace.analyze"):
    img_rgb = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
    res = DeepFace.analyze(
        img_path=img_rgb, actions=["emotion"],
        detector_backend=DEEPFACE_DETECTOR, enforce_detection=False, silent=True
    )
    if isinstance(res, list):
        res = res[0]
    dom = res.get("dominant_emotion", "neutral").lower()
    df_preds.append(NAME_TO_IDX[DEEPFACE_TO_CLASS.get(dom, "neutral")])
df_elapsed = time.perf_counter() - t0

df_preds = np.array(df_preds, dtype=np.int64)
np.save(OTHERS_DIR / "deepface_preds.npy", df_preds)
json.dump(
    {
        "n": int(len(y_df)),
        "elapsed_sec": float(df_elapsed),
        "ms_img": float((df_elapsed / len(y_df)) * 1000.0),
        "detector_backend": DEEPFACE_DETECTOR,
        "accuracy": float(accuracy_score(y_df, df_preds)),
        "f1_macro": float(f1_score(y_df, df_preds, average="macro", zero_division=0)),
    },
    open(OTHERS_DIR / "deepface_meta.json", "w"),
    indent=2
)
print(f"DeepFace -> Acc={accuracy_score(y_df, df_preds):.4f} | F1-macro={f1_score(y_df, df_preds, average='macro'):.4f} | {(df_elapsed/len(y_df))*1000:.2f} ms/img")

DeepFace.analyze:   0%|          | 0/1000 [00:00<?, ?it/s]

DeepFace -> Acc=0.5590 | F1-macro=0.5455 | 5.05 ms/img
